# Course 5: Building a Small Bangla Language Model
## Session 7: Evaluation I, the model itself

**Goal:** compute perplexity and bits per character, control sampling, and set up a blind rating.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network.

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def train_bpe(lines, vocab_size, specials=("[UNK]", "<|endoftext|>")):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.Metaspace()
    tok.decoder = decoders.Metaspace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=list(specials))
    tok.train_from_iterator(lines, trainer)
    return tok

In [ ]:
# A tiny model and tokenizer trained on the built-in sample, so this notebook runs anywhere.
# Replace with your own tokenizer file and checkpoint from sessions 4 and 6.
torch.manual_seed(0)
tok = train_bpe(SAMPLE_LINES * 20, 300)
EOT = tok.token_to_id("<|endoftext|>")
ids = []
for line in SAMPLE_LINES * 20:
    ids += tok.encode(line).ids + [EOT]
data = torch.tensor(ids)
cfg = Config(vocab_size=tok.get_vocab_size(), block_size=32, n_layer=2, n_head=2, d_model=64)
model = GPT(cfg)

def get_batch(data, block_size, batch_size):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(300):
    x, y = get_batch(data, cfg.block_size, 32)
    _, loss = model(x, y)
    opt.zero_grad(); loss.backward(); opt.step()
print("tiny model trained, final loss", round(loss.item(), 3))

### 1. Loss, perplexity and bits per character on held-out text
Replace `test_lines` with the lines of your `test.txt`.

In [ ]:
test_lines = SAMPLE_LINES[:4]            # TODO: lines of test.txt (here: seen text, so the score is flattering)

@torch.no_grad()
def score(model, tok, lines):
    model.eval()
    total_nll, n_tokens, n_chars = 0.0, 0, 0
    for line in lines:
        ids = [EOT] + tok.encode(line).ids + [EOT]
        ids = ids[:model.c.block_size + 1]
        x = torch.tensor([ids[:-1]]); y = torch.tensor([ids[1:]])
        logits, _ = model(x)
        nll = F.cross_entropy(logits[0], y[0], reduction="sum").item()
        total_nll += nll; n_tokens += len(ids) - 1; n_chars += len(line)
    mean_nll = total_nll / n_tokens
    return {"loss": mean_nll, "perplexity": math.exp(mean_nll),
            "bits_per_char": total_nll / math.log(2) / n_chars}

print(score(model, tok, test_lines))
print("check: loss 2.075 ->", round(math.exp(2.075), 2), "perplexity,", round(2.075 / math.log(2), 2), "bits per token")

### 2. Top-p sampling

In [ ]:
def sample_top_p(logits, p=0.9, temperature=1.0):
    probs = torch.softmax(logits / temperature, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_probs, dim=-1)
    keep = cum - sorted_probs < p
    sorted_probs = sorted_probs * keep
    sorted_probs = sorted_probs / sorted_probs.sum()
    return sorted_idx[torch.multinomial(sorted_probs, 1)].item()

logits = torch.log(torch.tensor([0.5, 0.3, 0.15, 0.05]))
print("p=0.9:", sorted({sample_top_p(logits, 0.9) for _ in range(500)}))
print("p=0.7:", sorted({sample_top_p(logits, 0.7) for _ in range(500)}))

### 3. Generate at three settings

In [ ]:
prompts = ["আমি", "বাংলা", "মা"]          # TODO: your 20 fixed prompts
for temperature, top_p in [(0.7, 0.9), (1.0, 0.9), (1.3, 1.0)]:
    print(f"--- temperature {temperature}, top_p {top_p}")
    for p in prompts:
        start = torch.tensor([tok.encode(p).ids])
        out = model.generate(start, 12, temperature=temperature, top_p=top_p)
        print("  ", tok.decode(out[0].tolist()))

### 4. Blind rating sheet

In [ ]:
import csv, random
random.seed(0)
rows = []
for p in prompts:
    for name in ["model_A", "model_B"]:          # TODO: generate from your two models
        out = model.generate(torch.tensor([tok.encode(p).ids]), 12)
        rows.append({"model": name, "prompt": p, "output": tok.decode(out[0].tolist())})
random.shuffle(rows)
with open("rating_sheet.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["id", "prompt", "output", "grammar_1to5", "coherence_1to5"])
    for i, r in enumerate(rows):
        w.writerow([i, r["prompt"], r["output"], "", ""])
with open("rating_key.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["id", "model"])
    for i, r in enumerate(rows):
        w.writerow([i, r["model"]])
print("wrote rating_sheet.csv (give to readers) and rating_key.csv (keep hidden)")

### TODO
- Have two readers fill in the sheet, then join it with the key and compute mean scores per model and how often the readers agree within one point.
- Collect five typical errors with examples.